# 👍🏻👎🏻📝📈 <span style="color: white; background-color: DodgerBlue"><b> Análise de Sentimento sobre a Pesquisa de Satisfação dos Associados nas UL's </b></span></p>

🖥️ <span style="color: DeepSkyBlue"><b> 1- Interface Gráfica de Configuração (Tkinter) </b></span></p>
- O usuário abre uma janela desktop profissional onde pode:
    - Selecionar o ano (2016 a 2026);
    - Selecionar mês(es) (multi-seleção, com botão "Selecionar todos");
    - Selecionar unidade(s) (24 unidades de lazer: Amparo, Campos do Jordão, Guarujá, Ubatuba, etc.);
    - Confirmar a execução com resumo automático: "Total de combinações: X"
- Essa camada torna a automação acessível a qualquer usuário do RH, sem necessidade de editar código

🕷️ <span style="color: DeepSkyBlue"><b> 2- Raspagem de Comentários (Selenium) </b></span></p>
- O scraper acessa o portal interno de qualidade:
    - http://apps/qualidade/satisfacao/v02/Comentarios_cc.aspx
- E executa automaticamente:
    - Seleção de filtros;
    - Unidade (dropdown);
    - Período: mês/ano inicial e final;
    - Marcação de todos os tipos de comentário (Elogio, Sugestão, Reclamação, Outros)
- Clique em "Exibir" e aguardar carregamento
- Usa WebDriverWait para detectar mudanças dinâmicas no container lyTabela
- Extração com 3 estratégias robustas
- Caso a estrutura HTML mude, o script tenta:
    -  table.comentarios (estrutura principal);
    - Divs dentro de #lyTabela;
    - Qualquer tabela genérica dentro de #lyTabela
- Paginação automática
- Detecta links de paginação, clica na próxima página e continua coletando até a última
- Detecção de resultados vazios
- Se não há comentários para uma combinação unidade/mês, gera um registro placeholder para manter a trilha de auditoria

💾 <span style="color: DeepSkyBlue"><b> 3- Persistência e Checkpoint </b></span></p>
- Salvamento parcial (JSON)
- A cada 10 combinações processadas, o script salva um checkpoint em comentarios_parcial.json, garantindo que nenhum dado seja perdido em caso de falha
- Salvamento final (Excel)
- Gera o arquivo comentarios_{mês}_{ano}.xlsx
- Com:
    - Aba "comentarios";
    - Largura automática de colunas (limite 60);
    - Quebra de texto habilitada (wrap_text=True);
    - Registros deduplicados (por unidade + período + conteúdo)

🧹 <span style="color: DeepSkyBlue"><b>  4- Limpeza de Metadados </b></span></p>
- Antes da análise de sentimentos, o script remove metadados administrativos que poluem os textos:
    - "Matrícula: 12345";
    - "Período: 06/2026";
    - "Respondido: 15/06/2026"
- Usando expressões regulares (re.sub), deixando apenas o comentário real do hóspede

🤖 <span style="color: DeepSkyBlue"><b> 5- Análise de Sentimentos — Léxico Customizado de Hospitalidade </b></span></p>
- Este é o diferencial estratégico do projeto. Em vez de depender apenas de modelos genéricos, o script utiliza:
    - Léxico Manual com +200 termos do setor de hospitalidade
- Palavras Positivas (exemplos):
    - "atencioso": 1.5, "café da manhã excelente": 2.0, "impecável": 2.0, "recomendo": 1.8
- Palavras Negativas (exemplos):
    - "mau atendimento": -2.0, "sujo": -1.8, "ar condicionado quebrado": -1.8, "propaganda enganosa": -2.0
- Combinação com VADER (NLTK)
- Se o NLTK estiver disponível, o score final é uma média ponderada:
    - score = (léxico x 0.6) + (VADER x 0.4)
- Isso garante que o modelo capture tanto o contexto de hospitalidade quanto a semântica geral da língua portuguesa
- Classificações geradas:
    - polaridade — score entre -1.0 e +1.0;
    - sentimento — positivo / neutro / negativo;
    - nps_categoria — Promotor / Neutro / Detrator

📈 <span style="color: DeepSkyBlue"><b> 6- Cálculo de NPS (Net Promoter Score) </b></span></p>
- O script calcula automaticamente:
    - NPS = (Promotores - Detratores / Total Válidos) x 100
- E classifica o resultado:
    - 🟢 Excelente — NPS ≥ 50;
    - 🟡 Razoável — NPS entre 0 e 49;
    - 🔴 Crítico — NPS < 0

📊 <span style="color: DeepSkyBlue"><b> 7- Visualizações Geradas (Matplotlib + WordCloud) </b></span></p>
- O relatório inclui 8+ visualizações:
    - Distribuição de Sentimentos (barra com percentuais);
    - Distribuição por Tipo (Elogio, Reclamação, Sugestão, Outros);
    - Ranking de Unidades por % de Positivos (barras horizontais empilhadas);
    - Tendência Mensal (linha temporal);
    - Nuvem de Palavras Geral;
    - Nuvem de Palavras — Comentários Positivos;
    - Nuvem de Palavras — Comentários Negativos;
    - Top 15 Palavras mais frequentes (positivas e negativas separadamente)
- Todas as imagens são convertidas para Base64 e embutidas diretamente no HTML

🌐 <span style="color: DeepSkyBlue"><b> 8- Relatório HTML Final Interativo </b></span></p>
- O script gera:
    - Relatorio_Sentimentos_Hospedes.html
- Com:
    - Design corporativo (cores AFPESP, logo embutida);
    - Cards de métricas (Total, Positivos, Neutros, Negativos);
    - Seção de NPS com score e classificação;
    - Tabela detalhada de unidades (com polaridade média);
    - Top 10 comentários mais positivos e mais negativos;
    - Botão de download do Excel completo;
    - Layout responsivo
- O relatório é aberto automaticamente no navegador ao final da execução

📦 <span style="color: DeepSkyBlue"><b> 9- Excel Estruturado Final </b></span></p>
- Gera também:
    - Dados_Sentimentos_Hospedes.xlsx
- Com:
    - Cabeçalho azul corporativo (#005A9C);
    - Colunas: unidade, mês, ano, período, tipo, conteúdo, texto limpo, polaridade, sentimento, NPS;
    Largura automática e freeze panes na linha 1

# Raspador de Comentários - Pesquisa de Satisfação

In [1]:
# ===================================================================
# IMPORTANDO AS BIBLIOTECAS
# ===================================================================

import json
import logging
import os
import time
from datetime import datetime

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait

import tkinter as tk
from tkinter import ttk, messagebox

NL = chr(10)

# Caminho fixo onde o arquivo sera salvo
CAMINHO_SAIDA = r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Sentimentos\Pesquisa de Satisfação\Bases de Comentários'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler('raspagem_comentarios.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

UNIDADES = [
    ('31203000', 'Amparo'),
    ('31219000', 'Appenzell'),
    ('31212000', 'Areado'),
    ('31215000', 'Avaré'),
    ('31231000', 'Boraceia'),
    ('31205000', 'Campos do Jordão'),
    ('31206000', 'Caraguatatuba'),
    ('31204000', 'Guarujá'),
    ('31217000', 'Ibirá'),
    ('31228000', 'Ibirá - Fazenda'),
    ('31218000', 'Itanhaém'),
    ('31210000', 'Lindoia'),
    ('31220000', 'Maresias'),
    ('31221000', 'Monte Verde'),
    ('31227000', 'Peruíbe I'),
    ('31226000', 'Peruíbe II'),
    ('31208000', 'Poços de Caldas'),
    ('31224000', 'Saha'),
    ('31229000', 'São Lourenço'),
    ('31216000', 'São Pedro'),
    ('31213000', 'Serra Negra'),
    ('31209000', 'Socorro'),
    ('31214000', 'Ubatuba'),
    ('31222000', 'Unidade Capital'),
]

MESES = [
    ('1', 'janeiro'), ('2', 'fevereiro'), ('3', 'março'),
    ('4', 'abril'), ('5', 'maio'), ('6', 'junho'),
    ('7', 'julho'), ('8', 'agosto'), ('9', 'setembro'),
    ('10', 'outubro'), ('11', 'novembro'), ('12', 'dezembro'),
]

ANOS = ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

# ===================================================================
# INTERFACE GRAFICA (Tkinter)
# ===================================================================

class ConfiguracaoGUI:
    """Janela grafica para selecionar ano, meses e unidades."""

    def __init__(self):
        self.resultado = None  # Sera preenchido ao confirmar
        self.janela = tk.Tk()
        self.janela.title('Raspagem de Comentarios — Pesquisa de Satisfacao')
        self.janela.geometry('500x680')
        self.janela.resizable(False, False)
        self.janela.configure(bg='#f0f0f0')

        self._construir_ui()

    def _construir_ui(self):
        # Titulo
        frame_titulo = tk.Frame(self.janela, bg='#2c3e50', height=50)
        frame_titulo.pack(fill='x')
        tk.Label(
            frame_titulo, text='Raspagem de Comentarios',
            font=('Segoe UI', 14, 'bold'), fg='white', bg='#2c3e50'
        ).pack(pady=12)

        # --- ANO ---
        frame_ano = tk.LabelFrame(self.janela, text='Ano', font=('Segoe UI', 10, 'bold'), padx=10, pady=5)
        frame_ano.pack(fill='x', padx=15, pady=(10, 5))

        self.var_ano = tk.StringVar(value='2026')
        ttk.Combobox(
            frame_ano, textvariable=self.var_ano, values=ANOS,
            state='readonly', width=10, font=('Segoe UI', 10)
        ).pack(anchor='w', pady=5)

        # --- MESES ---
        frame_meses = tk.LabelFrame(self.janela, text='Meses', font=('Segoe UI', 10, 'bold'), padx=10, pady=5)
        frame_meses.pack(fill='x', padx=15, pady=5)

        frame_meses_botoes = tk.Frame(frame_meses)
        frame_meses_botoes.pack(fill='x')

        tk.Button(frame_meses_botoes, text='Selecionar todos', command=self._sel_todos_meses,
                  font=('Segoe UI', 8)).pack(side='left', padx=(0, 5))
        tk.Button(frame_meses_botoes, text='Limpar', command=self._limpar_meses,
                  font=('Segoe UI', 8)).pack(side='left')

        self.listbox_meses = tk.Listbox(
            frame_meses, selectmode='multiple', height=6,
            font=('Segoe UI', 9), exportselection=False
        )
        for val, nome in MESES:
            self.listbox_meses.insert('end', nome)
        self.listbox_meses.pack(fill='x', pady=5)

        # --- UNIDADES ---
        frame_unidades = tk.LabelFrame(
            self.janela, text='Unidades', font=('Segoe UI', 10, 'bold'), padx=10, pady=5
        )
        frame_unidades.pack(fill='both', expand=True, padx=15, pady=5)

        frame_unid_botoes = tk.Frame(frame_unidades)
        frame_unid_botoes.pack(fill='x')
        tk.Button(frame_unid_botoes, text='Selecionar todas', command=self._sel_todas_unidades,
                  font=('Segoe UI', 8)).pack(side='left', padx=(0, 5))
        tk.Button(frame_unid_botoes, text='Limpar', command=self._limpar_unidades,
                  font=('Segoe UI', 8)).pack(side='left')

        self.listbox_unidades = tk.Listbox(
            frame_unidades, selectmode='multiple', height=10,
            font=('Segoe UI', 9), exportselection=False
        )
        for val, nome in UNIDADES:
            self.listbox_unidades.insert('end', nome)
        self.listbox_unidades.pack(fill='both', expand=True, pady=5)

        # --- BOTAO INICIAR ---
        tk.Button(
            self.janela, text='Iniciar Raspagem',
            command=self._confirmar,
            font=('Segoe UI', 11, 'bold'), bg='#27ae60', fg='white',
            activebackground='#219150', activeforeground='white',
            relief='flat', height=2, cursor='hand2'
        ).pack(fill='x', padx=15, pady=10)

    def _sel_todos_meses(self):
        self.listbox_meses.select_set(0, 'end')

    def _limpar_meses(self):
        self.listbox_meses.selection_clear(0, 'end')

    def _sel_todas_unidades(self):
        self.listbox_unidades.select_set(0, 'end')

    def _limpar_unidades(self):
        self.listbox_unidades.selection_clear(0, 'end')

    def _confirmar(self):
        ano = self.var_ano.get()

        idx_meses = self.listbox_meses.curselection()
        if not idx_meses:
            messagebox.showwarning('Atencao', 'Selecione pelo menos um mes.')
            return

        idx_unidades = self.listbox_unidades.curselection()
        if not idx_unidades:
            messagebox.showwarning('Atencao', 'Selecione pelo menos uma unidade.')
            return

        meses_sel = [MESES[i] for i in idx_meses]
        unidades_sel = [UNIDADES[i] for i in idx_unidades]

        total = len(unidades_sel) * len(meses_sel)
        confirmar = messagebox.askyesno(
            'Confirmar',
            f'Ano: {ano}\n'
            f'Meses: {len(meses_sel)} selecionado(s)\n'
            f'Unidades: {len(unidades_sel)} selecionada(s)\n'
            f'Total de combinacoes: {total}\n\n'
            f'Deseja iniciar?'
        )

        if confirmar:
            self.resultado = {
                'ano': ano,
                'meses': meses_sel,
                'unidades': unidades_sel,
            }
            self.janela.destroy()

    def mostrar(self):
        self.janela.mainloop()
        return self.resultado

# ===================================================================
# SCRAPER
# ===================================================================

class ComentariosScraper:

    def __init__(self, url, headless=True):
        self.url = url
        self.comentarios = []
        self.vistos = set()
        self.driver = self._criar_driver(headless)
        self.wait = WebDriverWait(self.driver, 20)

    def _criar_driver(self, headless):
        options = Options()
        if headless:
            options.add_argument('--headless=new')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        )
        return webdriver.Chrome(options=options)

    def abrir_pagina(self):
        logger.info('Abrindo pagina: %s', self.url)
        self.driver.get(self.url)
        self.wait.until(EC.presence_of_element_located((By.ID, 'ddlUL')))
        self.wait.until(EC.presence_of_element_located((By.ID, 'btnExibir')))
        logger.info('Pagina carregada. Filtros disponiveis.')

    def selecionar_unidade(self, value, nome):
        ddl = Select(self.driver.find_element(By.ID, 'ddlUL'))
        ddl.select_by_value(value)
        logger.info('Unidade: %s', nome)
        time.sleep(0.3)

    def selecionar_periodo(self, mes_de_val, ano_de, mes_ate_val, ano_ate):
        Select(self.driver.find_element(By.ID, 'ddlfmesi')).select_by_value(mes_de_val)
        Select(self.driver.find_element(By.ID, 'ctl00_cph_ddlfAnoi')).select_by_value(ano_de)
        Select(self.driver.find_element(By.ID, 'ddlfmesf')).select_by_value(mes_ate_val)
        Select(self.driver.find_element(By.ID, 'ctl00_cph_ddlfAnof')).select_by_value(ano_ate)

    def marcar_todos_tipos(self):
        for tipo_id in ['ftipo1', 'ftipo2', 'ftipo3', 'ftipo4']:
            cb = self.driver.find_element(By.ID, tipo_id)
            if not cb.is_selected():
                cb.click()

    def clicar_exibir_e_aguardar(self):
        ly_tabela = self.driver.find_element(By.ID, 'lyTabela')
        conteudo_anterior = ly_tabela.get_attribute('innerHTML')

        btn = self.driver.find_element(By.ID, 'btnExibir')
        self.driver.execute_script('arguments[0].click();', btn)

        try:
            self.wait.until(
                lambda d: d.find_element(By.ID, 'lyTabela').get_attribute('innerHTML') != conteudo_anterior
            )
        except Exception:
            pass

        time.sleep(1.5)
        logger.info('Resultados carregados.')

    # 
    # VERIFICACAO DE RESULTADOS VAZIOS
    # 

    def _verificar_sem_resultados(self, unidade_nome, mes_nome, ano):
        """
        Verifica se a pagina de resultados esta vazia (sem comentarios).
        Se estiver, retorna um registro placeholder com mensagem.
        """
        try:
            ly_tabela = self.driver.find_element(By.ID, 'lyTabela')
            texto = ly_tabela.text.strip()

            # Se o container esta vazio ou so tem texto de "nenhum resultado"
            if not texto or len(texto) < 15:
                return [{
                    'unidade': unidade_nome,
                    'mes': mes_nome,
                    'ano': ano,
                    'periodo': f'{mes_nome}/{ano}',
                    'tipo': 'Sem comentarios',
                    'conteudo': 'Sem comentarios nesse periodo informado',
                    'texto_completo': 'Sem comentarios nesse periodo informado',
                    'extraido_em': datetime.now().isoformat(),
                    'pagina': '0',
                }]

            # Verifica se ha apenas texto de "nenhum resultado encontrado"
            palavras_vazio = ['nenhum', 'sem resultado', 'sem comentarios', 'nao encontrado', 'vazio']
            texto_lower = texto.lower()
            for palavra in palavras_vazio:
                if palavra in texto_lower and len(texto) < 80:
                    return [{
                        'unidade': unidade_nome,
                        'mes': mes_nome,
                        'ano': ano,
                        'periodo': f'{mes_nome}/{ano}',
                        'tipo': 'Sem comentarios',
                        'conteudo': 'Sem comentarios nesse periodo informado',
                        'texto_completo': 'Sem comentarios nesse periodo informado',
                        'extraido_em': datetime.now().isoformat(),
                        'pagina': '0',
                    }]

            return None

        except Exception:
            return None

    # 
    # EXTRAÇÃO
    # 

    def extrair_comentarios_pagina(self, unidade_nome, mes_nome, ano):
        comentarios_pagina = []

        # ESTRATEGIA 1: table.comentarios
        tabelas = self.driver.find_elements(By.CSS_SELECTOR, 'table.comentarios')
        if tabelas:
            for tabela in tabelas:
                linhas = tabela.find_elements(By.TAG_NAME, 'tr')
                for linha in linhas:
                    celulas = linha.find_elements(By.TAG_NAME, 'td')
                    if not celulas:
                        continue
                    textos = [c.text.strip() for c in celulas if c.text.strip()]
                    if not textos:
                        continue
                    comentario = {
                        'unidade': unidade_nome,
                        'mes': mes_nome,
                        'ano': ano,
                        'periodo': f'{mes_nome}/{ano}',
                        'tipo': self._detectar_tipo(textos),
                        'conteudo': ' | '.join(textos),
                        'texto_completo': NL.join(textos),
                        'extraido_em': datetime.now().isoformat(),
                        'pagina': self._pagina_atual(),
                    }
                    comentarios_pagina.append(comentario)

        # ESTRATEGIA 2: divs dentro de #lyTabela
        if not comentarios_pagina:
            ly_tabela = self.driver.find_element(By.ID, 'lyTabela')
            divs = ly_tabela.find_elements(By.CSS_SELECTOR, 'div')
            for div in divs:
                texto = div.text.strip()
                if len(texto) > 10:
                    comentario = {
                        'unidade': unidade_nome,
                        'mes': mes_nome,
                        'ano': ano,
                        'periodo': f'{mes_nome}/{ano}',
                        'tipo': self._detectar_tipo([texto]),
                        'conteudo': texto,
                        'texto_completo': texto,
                        'extraido_em': datetime.now().isoformat(),
                        'pagina': self._pagina_atual(),
                    }
                    comentarios_pagina.append(comentario)

        # ESTRATEGIA 3: qualquer tabela em #lyTabela
        if not comentarios_pagina:
            ly_tabela = self.driver.find_element(By.ID, 'lyTabela')
            tabelas = ly_tabela.find_elements(By.TAG_NAME, 'table')
            for tabela in tabelas:
                linhas = tabela.find_elements(By.TAG_NAME, 'tr')
                for linha in linhas:
                    celulas = linha.find_elements(By.TAG_NAME, 'td')
                    if not celulas:
                        continue
                    textos = [c.text.strip() for c in celulas if c.text.strip()]
                    if not textos:
                        continue
                    comentario = {
                        'unidade': unidade_nome,
                        'mes': mes_nome,
                        'ano': ano,
                        'periodo': f'{mes_nome}/{ano}',
                        'tipo': self._detectar_tipo(textos),
                        'conteudo': ' | '.join(textos),
                        'texto_completo': NL.join(textos),
                        'extraido_em': datetime.now().isoformat(),
                        'pagina': self._pagina_atual(),
                    }
                    comentarios_pagina.append(comentario)

        return comentarios_pagina

    def _detectar_tipo(self, textos):
        texto_completo = ' '.join(textos).lower()
        if 'elogio' in texto_completo:
            return 'Elogio'
        elif 'sugest' in texto_completo:
            return 'Sugestao'
        elif 'reclam' in texto_completo:
            return 'Reclamacao'
        elif 'outro' in texto_completo:
            return 'Outros'
        return 'Nao identificado'

    def _pagina_atual(self):
        try:
            links_ativos = self.driver.find_elements(By.CSS_SELECTOR, '.paginacao a.on')
            if links_ativos:
                return links_ativos[0].text.strip()
        except Exception:
            pass
        return '1'

    # 
    # PAGINAÇÃO
    # 

    def _obter_links_paginacao(self):
        containers = self.driver.find_elements(By.CSS_SELECTOR, '.paginacao')
        if not containers:
            return [], -1

        links = containers[0].find_elements(By.TAG_NAME, 'a')
        if not links:
            return [], -1

        idx_ativo = -1
        for i, link in enumerate(links):
            classes = link.get_attribute('class') or ''
            if 'on' in classes:
                idx_ativo = i
                break

        return links, idx_ativo

    def navegar_proxima_pagina(self):
        links, idx_ativo = self._obter_links_paginacao()

        if not links or len(links) <= 1:
            return False

        if idx_ativo < 0:
            return False
        if idx_ativo >= len(links) - 1:
            logger.info('  >> Pagina %s e a ultima (%d total) — proxima unidade',
                        links[idx_ativo].text.strip(), len(links))
            return False

        proximo_link = links[idx_ativo + 1]

        try:
            pagina_atual_num = int(links[idx_ativo].text.strip())
            pagina_proxima_num = int(proximo_link.text.strip())
        except ValueError:
            return False

        if pagina_proxima_num <= pagina_atual_num:
            return False

        ly_tabela = self.driver.find_element(By.ID, 'lyTabela')
        conteudo_anterior = ly_tabela.get_attribute('innerHTML')

        try:
            self.driver.execute_script('arguments[0].click();', proximo_link)
        except Exception as e:
            logger.warning('  >> Erro ao clicar: %s', e)
            return False

        mudou = False
        try:
            self.wait.until(
                lambda d: d.find_element(By.ID, 'lyTabela').get_attribute('innerHTML') != conteudo_anterior
            )
            mudou = True
        except Exception:
            mudou = False

        if not mudou:
            return False

        time.sleep(1.0)
        logger.info('  >> Pagina %d -> %d', pagina_atual_num, pagina_proxima_num)
        return True

    def coletar_todas_paginas(self, unidade_nome, mes_nome, ano):
        pagina = 1
        comentarios_ultima = 0
        encontrou_resultados = False

        while True:
            logger.info('  Pagina %d — %s — %s/%s', pagina, unidade_nome, mes_nome, ano)
            comentarios_pagina = self.extrair_comentarios_pagina(unidade_nome, mes_nome, ano)

            if comentarios_pagina:
                encontrou_resultados = True

            comentarios_ultima = len(comentarios_pagina)

            for c in comentarios_pagina:
                chave = f"{c['unidade']}_{c['periodo']}_{c['conteudo'][:100]}"
                if chave not in self.vistos:
                    self.vistos.add(chave)
                    self.comentarios.append(c)

            if not self.navegar_proxima_pagina():
                break
            pagina += 1

        # Se nao encontrou nenhum comentario em nenhuma pagina, adiciona placeholder
        if not encontrou_resultados:
            logger.info('  >> Sem comentarios para %s em %s/%s', unidade_nome, mes_nome, ano)
            placeholder = {
                'unidade': unidade_nome,
                'mes': mes_nome,
                'ano': ano,
                'periodo': f'{mes_nome}/{ano}',
                'tipo': 'Sem comentarios',
                'conteudo': 'Sem comentarios nesse periodo informado',
                'texto_completo': 'Sem comentarios nesse periodo informado',
                'extraido_em': datetime.now().isoformat(),
                'pagina': '0',
            }
            chave = f"{placeholder['unidade']}_{placeholder['periodo']}_{placeholder['conteudo']}"
            if chave not in self.vistos:
                self.vistos.add(chave)
                self.comentarios.append(placeholder)

        logger.info('  %d pagina(s) — %d comentarios na ultima', pagina, comentarios_ultima)

    # 
    # PERSISTÊNCIA
    # 

    def _gerar_nome_arquivo(self, meses, ano):
        """
        Gera o nome do arquivo baseado nos meses e ano selecionados.
        """
        if len(meses) == 1:
            sufixo = meses[0][1]  # nome do mes
        elif len(meses) == 12:
            sufixo = 'todos'
        else:
            primeiro = meses[0][1]
            ultimo = meses[-1][1]
            sufixo = f'{primeiro}-{ultimo}'

        return f'comentarios_{sufixo}_{ano}.xlsx'

    def salvar_final(self, meses, ano):
        if not self.comentarios:
            logger.warning('Nenhum comentario coletado.')
            return

        df = pd.DataFrame(self.comentarios)
        df = df.drop_duplicates(
            subset=['unidade', 'periodo', 'conteudo'],
            keep='first'
        ).reset_index(drop=True)

        # Garante que a pasta existe
        os.makedirs(CAMINHO_SAIDA, exist_ok=True)

        # Gera nome do arquivo
        nome_arquivo = self._gerar_nome_arquivo(meses, ano)
        caminho_completo = os.path.join(CAMINHO_SAIDA, nome_arquivo)

        # Exporta para Excel
        with pd.ExcelWriter(caminho_completo, engine='openpyxl') as writer:
            df.to_excel(writer, index=False, sheet_name='comentarios')
            worksheet = writer.sheets['comentarios']

            # Largura automatica das colunas (com limite de 60)
            for i, col in enumerate(df.columns, 1):
                max_len = max(
                    df[col].astype(str).map(len).max(),
                    len(col)
                )
                worksheet.column_dimensions[
                    worksheet.cell(row=1, column=i).column_letter
                ].width = min(max_len + 2, 60)

            # Quebra de texto nas celulas
            from openpyxl.styles import Alignment
            for row in worksheet.iter_rows(min_row=2):
                for cell in row:
                    cell.alignment = Alignment(wrap_text=True, vertical='top')

        logger.info('=' * 55)
        logger.info(' RASPAGEM CONCLUIDA: %d registros', len(df))
        logger.info(' Arquivo: %s', caminho_completo)
        logger.info('=' * 55)

        return caminho_completo

    def salvar_parcial(self):
        path = 'comentarios_parcial.json'
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(self.comentarios, f, ensure_ascii=False, indent=2)
        logger.info('Checkpoint: %d registros em %s', len(self.comentarios), path)

    # 
    # EXECUÇÃO
    # 

    def executar(self, unidades, meses, ano):
        self.abrir_pagina()

        total = len(unidades) * len(meses)
        atual = 0

        for mes_val, mes_nome in meses:
            for unidade_val, unidade_nome in unidades:
                atual += 1
                logger.info('[%d/%d] %s — %s/%s', atual, total, unidade_nome, mes_nome, ano)

                try:
                    self.selecionar_unidade(unidade_val, unidade_nome)
                    self.selecionar_periodo(mes_val, ano, mes_val, ano)
                    self.marcar_todos_tipos()
                    self.clicar_exibir_e_aguardar()
                    self.coletar_todas_paginas(unidade_nome, mes_nome, ano)

                except Exception as e:
                    logger.error('Erro em %s/%s/%s: %s', unidade_nome, mes_nome, ano, e)
                    self.salvar_parcial()
                    try:
                        self.driver.get(self.url)
                        self.wait.until(EC.presence_of_element_located((By.ID, 'ddlUL')))
                        time.sleep(2)
                    except Exception:
                        logger.error('Falha ao recuperar pagina. Abortando.')
                        break

                if atual % 10 == 0:
                    self.salvar_parcial()

        return self.salvar_final(meses, ano)

    def fechar(self):
        self.driver.quit()
        logger.info('Navegador fechado.')

# ===================================================================
# EXECUÇÃO
# ===================================================================

if __name__ == '__main__':
    # Abre a interface grafica
    gui = ConfiguracaoGUI()
    config = gui.mostrar()

    if config is None:
        raise SystemExit('Operacao cancelada pelo usuario.')

    ano = config['ano']
    meses = config['meses']
    unidades = config['unidades']

    scraper = ComentariosScraper(
        url='http://apps/qualidade/satisfacao/v02/Comentarios_cc.aspx',
        headless=False
    )

    try:
        caminho_arquivo = scraper.executar(
            unidades=unidades,
            meses=meses,
            ano=ano,
        )
        # Mensagem final
        print(NL + '=' * 55)
        print(f'  Arquivo salvo em:')
        print(f'  {caminho_arquivo}')
        print('=' * 55)
    finally:
        scraper.fechar()

2026-08-11 13:02:24,445 [INFO] Abrindo pagina: http://apps/qualidade/satisfacao/v02/Comentarios_cc.aspx
2026-08-11 13:02:25,054 [INFO] Pagina carregada. Filtros disponiveis.
2026-08-11 13:02:25,061 [INFO] [1/24] Amparo — junho/2026
2026-08-11 13:02:25,127 [INFO] Unidade: Amparo
2026-08-11 13:02:27,315 [INFO] Resultados carregados.
2026-08-11 13:02:27,317 [INFO]   Pagina 1 — Amparo — junho/2026
2026-08-11 13:02:29,691 [INFO]   >> Pagina 1 -> 2
2026-08-11 13:02:29,693 [INFO]   Pagina 2 — Amparo — junho/2026
2026-08-11 13:02:31,794 [INFO]   >> Pagina 2 -> 3
2026-08-11 13:02:31,796 [INFO]   Pagina 3 — Amparo — junho/2026
2026-08-11 13:02:33,903 [INFO]   >> Pagina 3 -> 4
2026-08-11 13:02:33,912 [INFO]   Pagina 4 — Amparo — junho/2026
2026-08-11 13:02:36,177 [INFO]   >> Pagina 4 -> 5
2026-08-11 13:02:36,184 [INFO]   Pagina 5 — Amparo — junho/2026
2026-08-11 13:02:38,684 [INFO]   >> Pagina 5 -> 6
2026-08-11 13:02:38,686 [INFO]   Pagina 6 — Amparo — junho/2026
2026-08-11 13:02:39,441 [INFO]   


  Arquivo salvo em:
  X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Sentimentos\Pesquisa de Satisfação\Bases de Comentários\comentarios_junho_2026.xlsx


2026-08-11 13:06:10,729 [INFO] Navegador fechado.


 # Análise de Sentimentos — Pesquisa de Satisfação dos Hóspedes

In [2]:
import base64
import logging
import os
import re
import string
import sys
import webbrowser
from collections import Counter
from datetime import datetime
from io import BytesIO

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# NLTK (com fallback)
logging.info("Importando NLTK...")
try:
    import nltk
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "nltk.downloader", "stopwords", "vader_lexicon"],
        capture_output=True, timeout=60
    )
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    from nltk.corpus import stopwords
    NLTK_AVAILABLE = True
    sia = SentimentIntensityAnalyzer()
    STOPWORDS_PT = set(stopwords.words('portuguese'))
except Exception as e:
    logging.warning(f"NLTK indisponível: {e}")
    NLTK_AVAILABLE = False
    sia = None
    STOPWORDS_PT = {
        'a','o','e','de','da','do','em','um','uma','para','com','por','mas',
        'mais','ou','quando','muito','já','está','eu','também','só','pelo',
        'pela','até','isso','ela','entre','era','depois','sem','mesmo','aos',
        'ter','seus','pra','pro','tbm','tb','vc','vcs','q','p','n','tá','tô',
        'ta','to','né','que','se','na','no','nos','nas','como','foi','sou',
    }

try:
    from wordcloud import WordCloud
    WC_AVAILABLE = True
except Exception:
    WC_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler('analise_sentimentos_hospedes.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =====================================================================
# CAMINHOS
# =====================================================================
PASTA_DADOS = r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Sentimentos\Pesquisa de Satisfação\Bases de Comentários'
PASTA_DESTINO = r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Sentimentos\Pesquisa de Satisfação'
PATH_LOGO = r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Sentimentos\Logo AFPESP.png'

# =====================================================================
# LÉXICO CUSTOMIZADO — HOSPITALIDADE + GERAL
# =====================================================================

PALAVRAS_POSITIVAS = {
    # Atendimento
    'atencioso': 1.5, 'atenciosa': 1.5, 'atenciosos': 1.5, 'atenciosas': 1.5,
    'atendimento excelente': 2.0, 'ótimo atendimento': 2.0, 'bom atendimento': 1.5,
    'receptivo': 1.0, 'receptiva': 1.0, 'receptividade': 1.2,
    'cordial': 1.2, 'cordialidade': 1.2, 'gentil': 1.2, 'gentileza': 1.2,
    'prestativo': 1.5, 'prestativa': 1.5, 'prestativos': 1.5, 'prestativas': 1.5,
    'educado': 1.2, 'educada': 1.2, 'simpatia': 1.2, 'simpático': 1.2, 'simpática': 1.2,
    'acolhedor': 1.5, 'acolhedora': 1.5, 'acolhimento': 1.5,
    # Estrutura e conforto
    'confortável': 1.5, 'confortaveis': 1.5, 'confortáveis': 1.5, 'conforto': 1.2,
    'limpo': 1.5, 'limpa': 1.5, 'limpeza': 1.5, 'higiene': 1.2, 'higienizado': 1.2,
    'organizado': 1.2, 'organizada': 1.2, 'organização': 1.0, 
    'estrutura excelente': 1.8, 'boa estrutura': 1.5, 'infraestrutura': 0.8,
    'reformado': 1.0, 'reformada': 1.0, 'novinho': 1.2, 'nova': 0.5,
    'bem decorado': 1.0, 'agradável': 1.2, 'agradavel': 1.2, 'acolhedor': 1.2,
    # Serviços
    'café da manhã excelente': 2.0, 'café da manhã bom': 1.5, 'ótimo café': 1.8,
    'bufet': 0.5, 'buffet': 0.5, 'farto': 1.0, 'saboroso': 1.5, 'delicioso': 1.8,
    'piscina excelente': 1.5, 'boa piscina': 1.2, 'lazer': 0.8,
    'wi-fi': 0.5, 'wifi bom': 1.0, 'internet boa': 1.0, 'impecável': 2.0,
    'estacionamento': 0.5, 'segurança': 0.8, 'seguro': 1.0,
    # Avaliações gerais positivas
    'excelente': 2.0, 'excelentes': 2.0, 'ótimo': 1.8, 'ótima': 1.8, 'ótimos': 1.8, 'ótimas': 1.8, 'otima': 1.8,
    'maravilhoso': 2.0, 'maravilhosa': 2.0, 'maravilha': 2.0, 'maravilhas': 2.0, 'parabéns': 2.0, 'parabenizar': 2.0,
    'perfeito': 2.0, 'perfeita': 2.0, 'perfeitos': 2.0, 'elogiar': 2.0, 'elogiando': 2.0, 'amo': 2.0, 'parabens': 2.0,
    'muito bom': 1.5, 'muito boa': 1.5, 'muito bons': 1.5, 'muito boas': 1.5, 'espetacular': 2.0, 'espetaculares': 2.0,
    'recomendo': 1.8, 'recomendado': 1.5, 'recomendada': 1.5, 'super recomendo': 2.0,
    'voltaria': 1.5, 'voltei': 1.0, 'voltaremos': 1.5, 'sempre volto': 1.5, 'elogios': 1.0, 'elogio': 1.0,
    'satisfeito': 1.5, 'satisfeita': 1.5, 'satisfeitos': 1.5, 'satisfação': 1.2, 'gostamos': 1.0, 'gostou': 1.0,
    'agradeço': 1.5, 'agradecer': 1.5, 'agradecido': 1.5, 'agradecida': 1.5, 'obrigado': 1.2,
    'gratidão': 1.5, 'feliz': 1.2, 'felizes': 1.2, 'encantado': 1.5, 'encantada': 1.5,
    'surpreendeu': 1.5, 'surpresa agradável': 1.5, 'supera expectativas': 2.0, 'linda': 1.0, 'lindo': 1.0, 
    'custo-benefício': 0.8, 'vale a pena': 1.2, 'valeu': 1.0, 'sensacional': 2.0,
    'eficiente': 1.2, 'eficientes': 1.2, 'ágil': 1.0, 'agilidade': 1.0, 'nota': 1.0,
    'dedicado': 1.2, 'dedicada': 1.2, 'competente': 1.5, 'competentes': 1.5,
    'responsável': 1.0, 'responsáveis': 1.0, 'profissional': 0.8, 'profissionais': 0.8,
    'qualidade': 1.0, 'qualidades': 1.0, 'bem localizado': 1.2, 'boa localização': 1.2,
    'tranquilo': 1.0, 'tranquila': 1.0, 'paz': 1.0, 'descanso': 1.0,
    'preço justo': 1.0, 'justo': 0.8, 'razoável': 0.5, 'razoavel': 0.5,
    'bem': 0.3, 'bom': 1.0, 'boa': 1.0, 'bons': 1.0, 'boas': 1.0,
}

PALAVRAS_NEGATIVAS = {
    # Atendimento ruim
    'mau atendimento': -2.0, 'péssimo atendimento': -2.0, 'atendimento ruim': -1.8,
    'desatencioso': -1.5, 'desatenciosa': -1.5, 'mal educado': -1.5, 'mal educada': -1.5,
    'grosseiro': -1.8, 'grosseira': -1.8, 'grosseiros': -1.8, 'grosseiras': -1.8,
    'desrespeitoso': -1.5, 'desrespeitosa': -1.5, 'falta de educação': -1.5,
    'ignoraram': -1.5, 'ignorado': -1.2, 'ignorada': -1.2, 'ignorou': -1.5,
    'indiferença': -1.2, 'indiferente': -1.2, 'rude': -1.5, 
    'não quis ajudar': -1.8, 'não ajudou': -1.5, 'não resolveram': -1.5,
    'demora no atendimento': -1.5, 'demorado': -1.0, 'demorou': -0.8, 'lento': -1.0, 'lenta': -1.0,
    # Estrutura e conforto
    'sujo': -1.8, 'sujos': -1.8, 'suja': -1.8, 'sujas': -1.8, 'sujeira': -1.5,
    'mal cheiro': -1.5, 'mau cheiro': -1.5, 'cheiro ruim': -1.5, 'fedorento': -1.5,
    'desorganizado': -1.2, 'desorganizada': -1.2, 'bagunça': -1.2, 'bagunçado': -1.2,
    'quebrado': -1.5, 'quebrada': -1.5, 'estragado': -1.5, 'estragada': -1.5,
    'não funcionava': -1.5, 'não funciona': -1.5, 'fora de serviço': -1.5, 'duro': -0.5,
    'vazamento': -1.5, 'vazando': -1.5, 'infiltração': -1.5, 'infiltrou': -1.5,
    'ar condicionado quebrado': -1.8, 'ar não funcionava': -1.8, 'ar gelando': -1.0,
    'chuveiro frio': -1.5, 'sem água quente': -1.8, 'sem água': -1.8, 'pegajosa': -1.0, 'pegajoso': -1.0,
    'cama desconfortável': -1.5, 'colchão ruim': -1.5, 'cama dura': -1.2,
    'barulho': -1.2, 'barulhento': -1.5, 'muito barulho': -1.5, 'perturbou': -1.2,
    'sem janela': -0.8, 'escura': -0.5, 'abafado': -1.0, 'abafada': -1.0,
    'velho': -0.8, 'velha': -0.8, 'velhas': -0.8, 'velhos': -0.8, 'cansado': -0.5,
    'mal conservado': -1.5, 'degradado': -1.5, 'precisando reforma': -1.2,
    # Serviços
    'café da manhã ruim': -1.8, 'café da manhã fraco': -1.5, 'pouca opção': -1.0,
    'comida fria': -1.5, 'comida ruim': -1.8, 'sem gosto': -1.5, 'insosso': -1.2,
    'piscina suja': -1.8, 'piscina fechada': -1.5, 'sem piscina': -1.0, 'mais quente': -1.0,
    'wifi não funcionava': -1.5, 'internet ruim': -1.2, 'sem internet': -1.5, 'sem wifi': -1.5,
    'sem estacionamento': -1.0, 'estacionamento lotado': -1.0, 'caiu': -0.8,
    # Avaliações gerais negativas
    'péssimo': -2.0, 'péssima': -2.0, 'péssimos': -2.0, 'péssimas': -2.0,
    'horrível': -2.0, 'horripilante': -2.0, 'decepcionante': -1.8, 'decepção': -1.8,
    'ruim': -1.5, 'ruins': -1.5, 'pior': -1.5, 'pior hotel': -2.0, 'desagradável': -2.0,
    'não recomendo': -2.0, 'não recomendo': -2.0, 'nunca mais': -2.0,
    'arrependimento': -1.8, 'arrependi': -1.8, 'perda de tempo': -1.8, 'perda de dinheiro': -2.0,
    'insatisfeito': -1.5, 'insatisfeita': -1.5, 'insatisfação': -1.5,
    'reclamação': -1.0, 'reclamacao': -1.0, 'reclamei': -1.0, 'reclamaram': -1.0,
    'problema': -1.0, 'problemas': -1.0, 'dificuldade': -0.8, 'dificuldades': -0.8, 'difícil': -0.8,
    'falta de': -1.0, 'faltou': -0.8, 'sem': -0.3,
    'caro': -0.8, 'cara': -0.8, 'custoso': -0.8, 'não vale': -1.5,
    'enganação': -2.0, 'enganação': -2.0, 'propaganda enganosa': -2.0,
    'vergonha': -1.5, 'vergonhoso': -1.5, 'vergonhosa': -1.5,
    'fraude': -2.0, 'golpe': -2.0,'pena': -0.5,
    'aborto': -1.5, 'desastre': -1.8, 'catastrofe': -1.8, 'catástrofe': -1.8,
    'humilhado': -1.8, 'humilhada': -1.8, 'humilhação': -1.8,
    'descaso': -1.8, 'descaso com': -2.0, 'desprezo': -1.5, 'desprezar': -1.5,
    'desorganização': -1.2, 'improviso': -1.0, 'improvisado': -1.0,
    'feio': -0.8, 'feia': -0.8, 'feios': -0.8, 'feias': -0.8,
    'escuro': -0.5, 'frio': -0.8, 'gelado': -0.8,
    'desleixo': -1.5, 'abandonado': -1.5, 'abandono': -1.5,
    'inseguro': -1.5, 'insegurança': -1.5, 'perigoso': -1.8,
}

# Stopwords genéricas para WordCloud
PALAVRAS_GENERICAS = {
    'mesma','mesmo','mesmos','mesmas','gostaria','gostarias','gostariam','sobre', 'sem comentario', 'sem comentário',
    'sobres','onde','aonde','porém','porem','sempre','quando','assim','então','UL', 'sem comentários', 'sem comentarios',
    'entao','também','tambem','outro','melhor','agora','depois','antes','nunca',
    'ainda','muito','muita','muitos','muitas','pouco','pouca','bastante','apenas',
    'somente','só','já','talvez','tipo','tipos','pra','pois','coisa','coisas', 'sem nada a comentar',
    'forma','maneira','jeito','lado','ponto','parte','vez','vezes','fazer','ter',
    'momento','momentos','tempo','dia','afpesp','afp','bom','todos','todo','toda','unidade',
    'todas','alguma','algumas','algum','alguns','outra','outras','outro','outros','respondido',
    'cada','quanto','quanta','nada','tudo','algo','alguém','ninguém','matrícula','período',
}

# =====================================================================
# FUNÇÕES DE ANÁLISE
# =====================================================================

def get_polarity(text):
    """
    Análise de sentimento com léxico customizado + VADER.
    Retorna score entre -1.0 (muito negativo) e +1.0 (muito positivo).
    """
    if not isinstance(text, str) or not text.strip():
        return 0.0

    if 'Sem comentarios' in text or 'sem comentarios' in text.lower():
        return 0.0

    text_lower = text.lower()
    neg_score = 0.0
    pos_score = 0.0

    for word, peso in PALAVRAS_NEGATIVAS.items():
        if word in text_lower:
            neg_score += peso

    for word, peso in PALAVRAS_POSITIVAS.items():
        if word in text_lower:
            pos_score += peso

    # Se muito negativo, prevalece
    if neg_score <= -3.0:
        return -1.0
    if neg_score < -0.5:
        return max(-1.0, neg_score + pos_score * 0.3)
    if pos_score > 0.5:
        return min(1.0, pos_score - abs(neg_score) * 0.5)

    # Combina com VADER
    if NLTK_AVAILABLE and sia is not None:
        try:
            vader = sia.polarity_scores(text)["compound"]
            combined = (neg_score + pos_score) * 0.6 + vader * 0.4
            return max(-1.0, min(1.0, combined))
        except:
            pass

    final = neg_score + pos_score
    return max(-1.0, min(1.0, final / 10)) if final != 0 else 0.0

def classify_sentiment(score):
    """Classifica o score em categoria."""
    if score >= 0.1:
        return 'positivo'
    elif score <= -0.1:
        return 'negativo'
    return 'neutro'

def classify_nps(score):
    """Classifica em categoria NPS: Promotor / Neutro / Detrator."""
    if score >= 0.4:
        return 'Promotor'
    elif score <= -0.1:
        return 'Detrator'
    return 'Neutro'

def filter_words(words_list):
    """Filtra stopwords e palavras genéricas para WordCloud."""
    stopwords_full = STOPWORDS_PT.union(PALAVRAS_GENERICAS)
    return [
        word.lower()
        for word in words_list
        if (word.lower() not in stopwords_full and
            len(word) >= 3 and
            word.isalpha())
    ]

# =====================================================================
# FUNÇÕES DE VISUALIZAÇÃO
# =====================================================================

def image_to_base64(path):
    try:
        with open(path, 'rb') as f:
            return base64.b64encode(f.read()).decode('utf-8')
    except:
        return None

def extract_wordcloud_base64(texts, bg='white', colormap='viridis', width=900, height=450):
    if not WC_AVAILABLE or not texts:
        return ""
    text = " ".join([str(t) for t in texts if t])
    words = text.lower().translate(str.maketrans("", "", string.punctuation)).split()
    filtered = filter_words(words)
    if not filtered:
        return ""
    try:
        wc = WordCloud(
            width=width, height=height, background_color=bg,
            colormap=colormap, random_state=42,
            max_words=80, collocations=False
        ).generate_from_frequencies(Counter(filtered))
        buffer = BytesIO()
        wc.to_image().save(buffer, format='PNG')
        buffer.seek(0)
        return f"data:image/png;base64,{base64.b64encode(buffer.getvalue()).decode('utf-8')}"
    except Exception as e:
        logger.error(f"Erro WordCloud: {e}")
        return ""

def create_sentiment_chart(df):
    """Gráfico de distribuição de sentimentos."""
    if df.empty:
        return ""
    counts = df['sentimento'].value_counts().to_dict()
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('white')
    colors = {'positivo': '#28a745', 'negativo': '#dc3545', 'neutro': '#ffc107'}
    order = ['positivo', 'neutro', 'negativo']
    keys = [k for k in order if k in counts]
    vals = [counts[k] for k in keys]
    color_list = [colors.get(k, '#999') for k in keys]

    bars = ax.bar(keys, vals, color=color_list, edgecolor='#333', linewidth=2.5, alpha=0.85, width=0.6)
    for bar in bars:
        h = bar.get_height()
        pct = h / max(sum(vals), 1) * 100
        ax.text(bar.get_x() + bar.get_width()/2., h, f'{int(h)}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=13, fontweight='bold')
    ax.set_title('Distribuição de Sentimentos', fontsize=16, fontweight='bold', pad=20, color='#005a9c')
    ax.set_ylabel('Quantidade', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.2, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return _fig_to_base64(fig)

def create_unit_ranking_chart(df):
    """Ranking de unidades por % de positivos."""
    if df.empty or 'unidade' not in df.columns:
        return ""
    df_clean = df[df['sentimento'] != 'Sem comentarios'].copy()
    if df_clean.empty:
        return ""
    stats = df_clean.groupby('unidade')['sentimento'].agg(
        total='count',
        positivos=lambda x: (x == 'positivo').sum(),
        negativos=lambda x: (x == 'negativo').sum(),
        neutros=lambda x: (x == 'neutro').sum(),
    ).reset_index()
    stats['pct_pos'] = (stats['positivos'] / stats['total'] * 100).round(1)
    stats['pct_neg'] = (stats['negativos'] / stats['total'] * 100).round(1)
    stats = stats.sort_values('pct_pos', ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(6, len(stats) * 0.4)))
    fig.patch.set_facecolor('white')
    y_pos = range(len(stats))
    ax.barh(y_pos, stats['pct_pos'], color='#28a745', edgecolor='#333', linewidth=1, label='% Positivo', height=0.6)
    ax.barh(y_pos, stats['pct_neg'], left=stats['pct_pos'], color='#dc3545', edgecolor='#333', linewidth=1, label='% Negativo', height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(stats['unidade'], fontsize=10)
    ax.set_xlabel('Percentual (%)', fontsize=12, fontweight='bold')
    ax.set_title('Ranking de Unidades por Sentimento', fontsize=14, fontweight='bold', pad=20, color='#005a9c')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(axis='x', alpha=0.2, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for i, (_, row) in enumerate(stats.iterrows()):
        ax.text(row['pct_pos'] + 1, i, f"{row['pct_pos']:.0f}%", va='center', fontsize=9, fontweight='bold', color='#28a745')
    plt.tight_layout()
    return _fig_to_base64(fig)

def create_monthly_trend_chart(df):
    """Tendência mensal de sentimentos."""
    if df.empty or 'mes' not in df.columns:
        return ""
    month_order = ['janeiro','fevereiro','março','abril','maio','junho',
                   'julho','agosto','setembro','outubro','novembro','dezembro']
    df_clean = df[df['sentimento'] != 'Sem comentarios'].copy()
    df_clean['mes'] = pd.Categorical(df_clean['mes'], categories=month_order, ordered=True)
    monthly = df_clean.groupby(['mes', 'sentimento']).size().unstack(fill_value=0)
    for col in ['positivo', 'neutro', 'negativo']:
        if col not in monthly.columns:
            monthly[col] = 0
    monthly = monthly[['positivo', 'neutro', 'negativo']]
    if monthly.empty:
        return ""

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('white')
    monthly.plot(kind='line', ax=ax, color={'positivo': '#28a745', 'negativo': '#dc3545', 'neutro': '#ffc107'},
                 linewidth=2.5, marker='o', markersize=8)
    ax.set_title('Tendência Mensal de Sentimentos', fontsize=14, fontweight='bold', pad=20, color='#005a9c')
    ax.set_xlabel('Mês', fontsize=12, fontweight='bold')
    ax.set_ylabel('Quantidade', fontsize=12, fontweight='bold')
    ax.legend(title='Sentimento', fontsize=10)
    ax.grid(alpha=0.2, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    return _fig_to_base64(fig)

def create_type_chart(df):
    """Distribuição por tipo (Elogio, Sugestão, Reclamação, Outros)."""
    if df.empty or 'tipo' not in df.columns:
        return ""
    type_counts = df['tipo'].value_counts()
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('white')
    colors_map = {'Elogio': '#28a745', 'Reclamacao': '#dc3545', 'Sugestao': '#ffc107',
                  'Outros': '#17a2b8', 'Nao identificado': '#6c757d', 'Sem comentarios': '#adb5bd'}
    colors = [colors_map.get(t, '#6c757d') for t in type_counts.index]
    bars = ax.bar(range(len(type_counts)), type_counts.values, color=colors, edgecolor='#333', linewidth=2, alpha=0.85, width=0.6)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h, f'{int(h)}', ha='center', va='bottom', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(type_counts)))
    ax.set_xticklabels(type_counts.index, rotation=30, ha='right', fontsize=11)
    ax.set_title('Distribuição por Tipo de Comentário', fontsize=14, fontweight='bold', pad=20, color='#005a9c')
    ax.set_ylabel('Quantidade', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.2, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return _fig_to_base64(fig)

def create_top_words_chart(texts, title, color='#dc3545', top_n=15):
    if not texts:
        return ""
    text = "".join(texts).lower().translate(str.maketrans("", "", string.punctuation)).split()
    filtered = filter_words(text)
    freq = Counter(filtered).most_common(top_n)
    if not freq:
        return ""
    words, counts = zip(*freq)
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('white')
    bars = ax.barh(range(len(words)), counts, color=color, edgecolor='#333', linewidth=1.5, alpha=0.85)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=11)
    ax.invert_yaxis()
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.3, bar.get_y() + bar.get_height()/2., f'{int(w)}', va='center', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20, color='#005a9c')
    ax.set_xlabel('Frequência', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.2, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return _fig_to_base64(fig)

def _fig_to_base64(fig):
    buffer = BytesIO()
    plt.savefig(buffer, format='png', dpi=120, bbox_inches='tight', facecolor='white')
    buffer.seek(0)
    img_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
    plt.close(fig)
    return f"data:image/png;base64,{img_str}"

def img_tag(b64, txt="Gráfico"):
    return f'<img src="{b64}" alt="{txt}">' if b64 and b64.startswith("data:image") else f'<p style="color:#999;text-align:center;padding:20px;">{txt} não disponível</p>'

def limpar_texto_comentario(texto):
    """
    Remove metadados administrativos do texto do comentario.
    Elimina: Matricula, Periodo, Respondido e variantes.
    """
    if not isinstance(texto, str) or not texto.strip():
        return ''
    
    import re
    
    texto_limpo = texto
    
    # Remove linhas que comecam com os campos de metadata
    # Padroes: "Matricula: 12345", "Período: 06/2026", "Respondido: 15/06/2026"
    padroes_metadata = [
        r'[Mm]atr[ií]cula\s*:?\s*\S+',
        r'[Pp]er[ií]odo\s*:?\s*\S+(?:\s*/\s*\S+)?',
        r'[Rr]espondido\s*:?\s*\S+(?:\s*/\s*\S+)?',
    ]
    
    for padrao in padroes_metadata:
        texto_limpo = re.sub(padrao, '', texto_limpo)
    
    # Remove a palavra isolada "Matricula", "Periodo", "Respondido" se restaram
    palavras_remover = [
        'Matricula', 'matricula', 'Matrícula', 'matrícula',
        'Periodo', 'periodo', 'Período', 'período',
        'Respondido', 'respondido',
    ]
    for palavra in palavras_remover:
        texto_limpo = texto_limpo.replace(palavra, '')
    
    # Remove caracteres residuais (| sobrando, : sobrando, espacosmultiplos)
    texto_limpo = re.sub(r'\s*\|\s*', ' ', texto_limpo)
    texto_limpo = re.sub(r'\s*:\s*', ' ', texto_limpo)
    texto_limpo = re.sub(r'\s{2,}', ' ', texto_limpo)
    texto_limpo = texto_limpo.strip()
    
    # Se ficou vazio apos limpeza, era so metadata
    if not texto_limpo or len(texto_limpo) < 3:
        return ''
    
    return texto_limpo

# =====================================================================
# PIPELINE PRINCIPAL
# =====================================================================

def main():
    logger.info("=" * 60)
    logger.info("ANÁLISE DE SENTIMENTOS — PESQUISA DE SATISFAÇÃO DE HÓSPEDES")
    logger.info("=" * 60)

    # 1. Localizar arquivo de dados
    arquivos = [f for f in os.listdir(PASTA_DADOS) if f.startswith('comentarios_') and f.endswith('.xlsx')]
    if not arquivos:
        logger.error(f"Nenhum arquivo comentarios_*.xlsx encontrado em {PASTA_DADOS}")
        return

    arquivos.sort(key=lambda x: os.path.getmtime(os.path.join(PASTA_DADOS, x)), reverse=True)
    arquivo_mais_recente = arquivos[0]
    path_dados = os.path.join(PASTA_DADOS, arquivo_mais_recente)
    logger.info(f"Arquivo de dados: {arquivo_mais_recente}")

    # 2. Carregar dados
    df = pd.read_excel(path_dados, engine='openpyxl')
    logger.info(f"{len(df)} registros carregados")

    # 3. LIMPEZA DOS METADADOS — cria coluna texto_limpo
    logger.info("Limpando metadados (Matricula, Periodo, Respondido)...")
    df['texto_limpo'] = df['texto_completo'].apply(limpar_texto_comentario)
    
    # Registros que ficaram vazios apos limpeza eram so metadata
    vazios = (df['texto_limpo'] == '').sum()
    if vazios > 0:
        logger.info(f"  {vazios} registro(s) continham apenas metadados (sem comentario real)")
    
    # Marca os que nao tem comentario real
    df.loc[df['texto_limpo'] == '', 'texto_limpo'] = 'Sem comentarios nesse periodo informado'
    
    # Log de exemplo para verificacao
    if len(df) > 0:
        exemplo = df.iloc[0]
        logger.info(f"  Exemplo original: {str(exemplo['texto_completo'])[:80]}...")
        logger.info(f"  Exemplo limpo:    {str(exemplo['texto_limpo'])[:80]}...")

    # 4. Análise de sentimento — USA texto_limpo em vez de texto_completo
    logger.info("Analisando sentimentos...")
    df['polaridade'] = df['texto_limpo'].apply(get_polarity)
    df['sentimento'] = df['polaridade'].apply(classify_sentiment)
    df['nps_categoria'] = df['polaridade'].apply(classify_nps)

    # 5. Métricas gerais
    df_validos = df[df['sentimento'] != 'Sem comentarios'].copy()
    total = len(df)
    total_validos = len(df_validos)
    pos = (df_validos['sentimento'] == 'positivo').sum()
    neu = (df_validos['sentimento'] == 'neutro').sum()
    neg = (df_validos['sentimento'] == 'negativo').sum()
    pct_pos = pos / max(total_validos, 1) * 100
    pct_neg = neg / max(total_validos, 1) * 100
    pct_neu = neu / max(total_validos, 1) * 100

    # NPS Score: % Promotores - % Detratores
    promotores = (df_validos['nps_categoria'] == 'Promotor').sum()
    detratores = (df_validos['nps_categoria'] == 'Detrator').sum()
    nps_score = round((promotores - detratores) / max(total_validos, 1) * 100)

    logger.info(f"Positivos: {pos} ({pct_pos:.1f}%) | Neutros: {neu} ({pct_neu:.1f}%) | Negativos: {neg} ({pct_neg:.1f}%)")
    logger.info(f"NPS Score: {nps_score}")

    # 6. Salvar XLSX estruturado
    caminho_xlsx = os.path.join(PASTA_DESTINO, 'Dados_Sentimentos_Hospedes.xlsx')
    df_saida = df[['unidade', 'mes', 'ano', 'periodo', 'tipo', 'conteudo',
                   'texto_completo', 'texto_limpo',
                   'polaridade', 'sentimento', 'nps_categoria',
                   'extraido_em', 'pagina']].copy()

    with pd.ExcelWriter(caminho_xlsx, engine='openpyxl') as writer:
        df_saida.to_excel(writer, index=False, sheet_name='Sentimentos')
        ws = writer.sheets['Sentimentos']
        from openpyxl.styles import PatternFill, Font, Alignment
        header_fill = PatternFill(start_color='005A9C', end_color='005A9C', fill_type='solid')
        header_font = Font(bold=True, color='FFFFFF', size=11)
        for col_idx in range(1, len(df_saida.columns) + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        ws.freeze_panes = 'A2'
        for col_idx, col_name in enumerate(df_saida.columns, 1):
            max_len = min(max(df_saida[col_name].astype(str).map(len).max(), len(col_name)) + 2, 60)
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = max_len
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical='top')
    logger.info(f"XLSX salvo: {caminho_xlsx}")

    # 7. Gerar visualizações
    logger.info("Gerando visualizações...")
    logo = image_to_base64(PATH_LOGO)

    # Usa texto_limpo em vez de texto_completo
    textos = df_validos['texto_limpo'].astype(str).tolist()
    wc_geral = extract_wordcloud_base64(textos, bg='white', colormap='viridis')
    chart_sentimento = create_sentiment_chart(df_validos)
    chart_unidades = create_unit_ranking_chart(df_validos)
    chart_mensal = create_monthly_trend_chart(df_validos)
    chart_tipo = create_type_chart(df_validos)

    pos_texts = df_validos[df_validos['sentimento'] == 'positivo']['texto_limpo'].astype(str).tolist()
    neg_texts = df_validos[df_validos['sentimento'] == 'negativo']['texto_limpo'].astype(str).tolist()
    wc_pos = extract_wordcloud_base64(pos_texts, bg='white', colormap='Greens')
    wc_neg = extract_wordcloud_base64(neg_texts, bg='#1a1a2e', colormap='Reds')
    chart_top_pos = create_top_words_chart(pos_texts, 'Top 15 Palavras — Comentários Positivos', color='#28a745')
    chart_top_neg = create_top_words_chart(neg_texts, 'Top 15 Palavras — Comentários Negativos', color='#dc3545')

    # 8. Tabela de comentários (top 10 mais negativos)
    top_neg = df_validos.nsmallest(10, 'polaridade')[['unidade', 'periodo', 'tipo', 'texto_limpo', 'polaridade']].to_dict('records')
    top_pos = df_validos.nlargest(10, 'polaridade')[['unidade', 'periodo', 'tipo', 'texto_limpo', 'polaridade']].to_dict('records')

    # 9. Estatísticas por unidade
    stats_unidades = df_validos.groupby('unidade').agg(
        total=('sentimento', 'count'),
        positivos=('sentimento', lambda x: (x == 'positivo').sum()),
        neutros=('sentimento', lambda x: (x == 'neutro').sum()),
        negativos=('sentimento', lambda x: (x == 'negativo').sum()),
        polaridade_media=('polaridade', 'mean'),
    ).reset_index()
    stats_unidades['pct_pos'] = (stats_unidades['positivos'] / stats_unidades['total'] * 100).round(1)
    stats_unidades['polaridade_media'] = stats_unidades['polaridade_media'].round(3)
    stats_unidades = stats_unidades.sort_values('polaridade_media', ascending=False)

    # 10. NPS classificação
    if nps_score >= 50:
        nps_class = "Excelente"
        nps_color = "#28a745"
    elif nps_score >= 0:
        nps_class = "Razoável"
        nps_color = "#ffc107"
    else:
        nps_class = "Crítico"
        nps_color = "#dc3545"

    # 10. Gerar HTML
    logger.info("Gerando relatório HTML...")

    # Tabela de unidades (HTML)
    tabela_unidades_html = ""
    for _, row in stats_unidades.iterrows():
        pol_color = "#28a745" if row['polaridade_media'] > 0.1 else ("#dc3545" if row['polaridade_media'] < -0.1 else "#ffc107")
        tabela_unidades_html += f"""
        <tr>
            <td>{row['unidade']}</td>
            <td style="text-align:center">{int(row['total'])}</td>
            <td style="text-align:center;color:#28a745;font-weight:bold">{int(row['positivos'])}</td>
            <td style="text-align:center;color:#ffc107;font-weight:bold">{int(row['neutros'])}</td>
            <td style="text-align:center;color:#dc3545;font-weight:bold">{int(row['negativos'])}</td>
            <td style="text-align:center">{row['pct_pos']:.1f}%</td>
            <td style="text-align:center;color:{pol_color};font-weight:bold">{row['polaridade_media']:.3f}</td>
        </tr>"""

    # Tabela top negativos
    tabela_neg_html = ""
    for item in top_neg:
        texto_curto = str(item['texto_limpo'])[:200] + "..." if len(str(item['texto_limpo'])) > 200 else str(item['texto_limpo'])
        tabela_neg_html += f"""
        <tr>
            <td>{item['unidade']}</td>
            <td>{item['periodo']}</td>
            <td>{item['tipo']}</td>
            <td style="font-size:0.85em">{texto_curto}</td>
            <td style="text-align:center;color:#dc3545;font-weight:bold">{item['polaridade']:.3f}</td>
        </tr>"""

    # Tabela top positivos
    tabela_pos_html = ""
    for item in top_pos:
        texto_curto = str(item['texto_limpo'])[:200] + "..." if len(str(item['texto_limpo'])) > 200 else str(item['texto_limpo'])
        tabela_pos_html += f"""
        <tr>
            <td>{item['unidade']}</td>
            <td>{item['periodo']}</td>
            <td>{item['tipo']}</td>
            <td style="font-size:0.85em">{texto_curto}</td>
            <td style="text-align:center;color:#28a745;font-weight:bold">{item['polaridade']:.3f}</td>
        </tr>"""

    logo_html = f'<img src="data:image/png;base64,{logo}" alt="Logo" style="height:60px;margin-bottom:10px;">' if logo else ""

    html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <title>Análise de Sentimentos — Pesquisa de Satisfação de Hóspedes</title>
    <style>
        * {{ margin:0; padding:0; box-sizing:border-box; }}
        body {{ font-family:'Segoe UI',sans-serif; background:#f0f2f5; color:#333; line-height:1.6; }}
        .container {{ max-width:1200px; margin:0 auto; padding:20px; }}
        header {{ background:linear-gradient(135deg,#005a9c,#003d6b); color:white; padding:40px; text-align:center; border-radius:12px; margin-bottom:30px; box-shadow:0 4px 12px rgba(0,0,0,0.15); }}
        header h1 {{ font-size:2.2em; margin-bottom:8px; }}
        header p {{ opacity:0.9; font-size:1.1em; }}
        .section {{ background:white; padding:30px; margin-bottom:25px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08); }}
        .section h2 {{ color:#005a9c; border-bottom:3px solid #28a745; padding-bottom:12px; margin-bottom:20px; font-size:1.5em; }}
        .metrics {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:15px; margin-bottom:30px; }}
        .metric {{ padding:20px; border-radius:10px; text-align:center; border-left:5px solid #6c757d; background:#f8f9fa; transition:transform 0.2s; }}
        .metric:hover {{ transform:translateY(-3px); box-shadow:0 4px 12px rgba(0,0,0,0.1); }}
        .metric.pos {{ border-left-color:#28a745; background:#f0fdf4; }}
        .metric.neg {{ border-left-color:#dc3545; background:#fdf0f0; }}
        .metric.neu {{ border-left-color:#ffc107; background:#fffdf0; }}
        .metric.nps {{ border-left-color:#6f42c1; background:#f8f0fc; }}
        .metric .value {{ font-size:2.2em; font-weight:bold; color:#005a9c; margin:8px 0; }}
        .metric.pos .value {{ color:#28a745; }}
        .metric.neg .value {{ color:#dc3545; }}
        .metric.neu .value {{ color:#ffc107; }}
        .metric .label {{ font-size:0.9em; color:#666; text-transform:uppercase; letter-spacing:0.5px; }}
        .chart {{ margin:20px 0; text-align:center; }}
        .chart img {{ max-width:100%; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.1); }}
        .btn {{ display:inline-block; background:#28a745; color:white; padding:12px 30px; border-radius:6px; text-decoration:none; font-weight:bold; margin:5px 0; transition:background 0.2s; }}
        .btn:hover {{ background:#218838; }}
        table {{ width:100%; border-collapse:collapse; margin:15px 0; font-size:0.9em; }}
        th {{ background:#005a9c; color:white; padding:12px 10px; text-align:left; font-weight:bold; }}
        td {{ padding:10px; border-bottom:1px solid #e0e0e0; }}
        tr:nth-child(even) {{ background:#f8f9fa; }}
        tr:hover {{ background:#e8f4f8; }}
        .badge {{ display:inline-block; padding:3px 12px; border-radius:20px; font-size:0.85em; font-weight:bold; color:white; }}
        .footer {{ text-align:center; padding:30px; color:#888; border-top:1px solid #ddd; margin-top:30px; font-size:0.9em; }}
        .nps-box {{ display:flex; align-items:center; justify-content:center; gap:15px; }}
        .nps-score {{ font-size:3em; font-weight:bold; }}
        .nps-label {{ font-size:1.2em; font-weight:bold; padding:5px 15px; border-radius:6px; color:white; }}
    </style>
</head>
<body>
<div class="container">
    <header>
        {logo_html}
        <h1>Análise de Sentimentos — Pesquisa de Satisfação</h1>
        <p>Hóspedes — Relatório de People Analytics — AFPESP</p>
        <p style="font-size:0.85em;margin-top:5px;">Gerado em {datetime.now().strftime('%d/%m/%Y às %H:%M')}</p>
    </header>

    <div class="section">
        <h2>Downloads</h2>
        <a href="file:///{caminho_xlsx.replace(os.sep, '/')}" class="btn">Baixar Dados Completos (XLSX)</a>
    </div>

    <div class="metrics">
        <div class="metric"><div class="label">Total de Registros</div><div class="value">{total}</div></div>
        <div class="metric pos"><div class="label">Positivos</div><div class="value">{pos}</div><p>{pct_pos:.1f}%</p></div>
        <div class="metric neu"><div class="label">Neutros</div><div class="value">{neu}</div><p>{pct_neu:.1f}%</p></div>
        <div class="metric neg"><div class="label">Negativos</div><div class="value">{neg}</div><p>{pct_neg:.1f}%</p></div>
    </div>

    <div class="section">
        <h2>Net Promoter Score (NPS)</h2>
        <div class="nps-box">
            <div class="nps-score" style="color:{nps_color}">{nps_score}</div>
            <div class="nps-label" style="background:{nps_color}">{nps_class}</div>
        </div>
        <p style="text-align:center;margin-top:15px;color:#666;">
            Promotores: {promotores} | Neutros: {total_validos - promotores - detratores} | Detratores: {detratores}
        </p>
    </div>

    <div class="section">
        <h2>Distribuição de Sentimentos</h2>
        <div class="chart">{img_tag(chart_sentimento)}</div>
    </div>

    <div class="section">
        <h2>Distribuição por Tipo</h2>
        <div class="chart">{img_tag(chart_tipo)}</div>
    </div>

    <div class="section">
        <h2>Ranking de Unidades</h2>
        <div class="chart">{img_tag(chart_unidades)}</div>
        <table>
            <thead><tr><th>Unidade</th><th>Total</th><th>Positivos</th><th>Neutros</th><th>Negativos</th><th>% Positivo</th><th>Polaridade Média</th></tr></thead>
            <tbody>{tabela_unidades_html}</tbody>
        </table>
    </div>

    <div class="section">
        <h2>Tendência Mensal</h2>
        <div class="chart">{img_tag(chart_mensal)}</div>
    </div>

    <div class="section">
        <h2>Nuvem de Palavras — Geral</h2>
        <div class="chart">{img_tag(wc_geral)}</div>
    </div>

    <div class="section">
        <h2>Comentários Positivos ({len(pos_texts)})</h2>
        <div class="chart">{img_tag(wc_pos)}</div>
        <div class="chart">{img_tag(chart_top_pos)}</div>
    </div>

    <div class="section">
        <h2>Comentários Negativos ({len(neg_texts)})</h2>
        <div class="chart">{img_tag(wc_neg)}</div>
        <div class="chart">{img_tag(chart_top_neg)}</div>
    </div>

    <div class="section">
        <h2>Top 10 Comentários Mais Positivos</h2>
        <table>
            <thead><tr><th>Unidade</th><th>Período</th><th>Tipo</th><th>Comentário</th><th>Polaridade</th></tr></thead>
            <tbody>{tabela_pos_html}</tbody>
        </table>
    </div>

    <div class="section">
        <h2>Top 10 Comentários Mais Negativos</h2>
        <table>
            <thead><tr><th>Unidade</th><th>Período</th><th>Tipo</th><th>Comentário</th><th>Polaridade</th></tr></thead>
            <tbody>{tabela_neg_html}</tbody>
        </table>
    </div>

    <div class="footer">
        <p>Relatório gerado automaticamente — People Analytics — AFPESP © 2026</p>
    </div>
</div>
</body>
</html>"""

    caminho_html = os.path.join(PASTA_DESTINO, 'Relatorio_Sentimentos_Hospedes.html')
    with open(caminho_html, 'w', encoding='utf-8') as f:
        f.write(html)
    logger.info(f"HTML salvo: {caminho_html}")

    try:
        webbrowser.open(f'file:///{caminho_html.replace(os.sep, "/")}')
    except:
        pass

    logger.info("=" * 60)
    logger.info("ANÁLISE CONCLUÍDA")
    logger.info(f"  Registros analisados: {total_validos}")
    logger.info(f"  NPS Score: {nps_score} ({nps_class})")
    logger.info(f"  HTML: {caminho_html}")
    logger.info(f"  XLSX: {caminho_xlsx}")
    logger.info("=" * 60)

if __name__ == '__main__':
    main()

KeyboardInterrupt: 